# 추천 모듈 개발 실험

[Colab에서 열기](https://colab.research.google.com/github/kimdonguk03/J1-travel-photo-curation/blob/main/notebooks/experiments.ipynb)

Google Drive에 올린 사진으로 추천 모듈을 점검하고 평가하는 노트북입니다.

| 섹션 | 내용 |
|---|---|
| 1. 처리 시간 | 모델 로딩, 원본 디코딩, CLIP 추론 시간을 따로 측정 |
| 2. 기준값 점검 | 묶음 확인(`clusters.html`)과 사진 쌍 표시(`pairs.html`)로 유사도 기준값 점검 |
| 3. 정답 페이지 | 평가 앨범 주인에게 보낼 `label_*.html` 생성 |
| 4. 추천 결과 예시 | 서버 연동 확인용 `recommend()` 결과 JSON |
| 5. 비교 평가 | 규칙을 고정한 뒤 owner.csv로 추천 방식 비교 |
| 6. 초안 평가 | 주인이 초안 사진을 '건질 사진 / 흔들림·흐림 / 별로'로 표시해 수용률 계산 |
| 7. 외부 검증 | Photo Triage 사람 선호 데이터로 연사 대표 컷 고르기를 검증·학습 |

- **처리 시간:** Colab에서 잰 값은 참고용입니다. 발표에 쓸 시간은 실제 시연 PC에서 다시 잽니다.
- **사진:** 사용 동의를 받은 사진만 올립니다.
- **HTML 결과:** 내려받아 내 PC의 브라우저로 엽니다. Colab 화면 안에서는 표시 저장과 내보내기가 막힐 수 있습니다.

## 0. 준비

저장소가 비공개라서 clone이 안 되면, GitHub에서 Code > Download ZIP으로 받아 Colab에 올립니다. 그다음 `/content/J1-travel-photo-curation`에 풀면 아래 셀이 그대로 동작합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
REPO = "/content/J1-travel-photo-curation"
if not os.path.exists(REPO):
    !git clone -q https://github.com/kimdonguk03/J1-travel-photo-curation.git {REPO}
else:
    !git -C {REPO} pull -q
%cd {REPO}/model
!pip install -q -r requirements.txt
!python tests/test_rules.py

Drive 폴더 경로를 적습니다. 평가 앨범은 `album_id: (사진 폴더, owner.csv 경로)` 형식이고, owner.csv는 앨범 주인에게 받은 뒤 채웁니다.

In [ ]:
J1 = "/content/drive/MyDrive/j1"            # Drive 작업 폴더
DEV_ALBUM = f"{J1}/dev_album"               # 개발 사진 (약 30장)
TEST_ALBUM = f"{J1}/test_album"             # 서버 연동 확인용 작은 앨범
EVAL = {                                     # 평가 앨범: album_id -> (사진 폴더, owner.csv)
    "gyeongju_trip": (f"{J1}/Gyeongju_trip", f"{J1}/owner/owner_gyeongju_trip.csv"),
    "unist_trip":    (f"{J1}/Unist_trip",    f"{J1}/owner/owner_unist_trip.csv"),
}
OUT = "/content/out"

import glob, json, shutil, statistics, sys, time
from pathlib import Path
sys.path.insert(0, ".")
import recommend as R
import j1_image as J
from curation import album_out_dir
from google.colab import files

for p in [DEV_ALBUM, TEST_ALBUM, *[a for a, _ in EVAL.values()]]:
    print(("있음  " if os.path.isdir(p) else "없음  ") + p)

**앨범 확인.** 앨범마다 사진 수와 촬영 시각이 없는 사진 수만 셉니다. 추천 결과는 보여주지 않으므로 평가 앨범에 실행해도 됩니다. '촬영 시각 미상'이 거의 전부라면 메신저로 받은 사진일 가능성이 높으니 원본으로 다시 받습니다.

In [ ]:
albums = {"dev_album": DEV_ALBUM, "test_album": TEST_ALBUM, **{k: a for k, (a, _) in EVAL.items()}}
print("| 앨범 | 사진 | 동영상 | 읽기 실패 | 촬영 시각 미상 | 형식 |")
print("|---|---|---|---|---|---|")
for name, folder in albums.items():
    if not os.path.isdir(folder):
        print(f"| {name} | 폴더 없음 | | | | |")
        continue
    ph, vid, bad = R.photos_from_folder(folder)
    exts = sorted({Path(p["path"]).suffix.lower() for p in ph})
    print(f"| {name} | {len(ph)} | {vid} | {len(bad)} | {sum(p['taken_at'] is None for p in ph)} | {' '.join(exts)} |")

## 1. 처리 시간

회차마다 캐시를 지우고 모델도 새로 올려 처음부터 계산합니다. 원본 디코딩은 앱과 같이 4개 스레드로 병렬 처리합니다. 1회차의 모델 로딩에는 모델 다운로드(약 89MB)가 섞일 수 있어 환산에서 뺍니다.

In [ ]:
photos, videos, failed = R.photos_from_folder(DEV_ALBUM)
unknown = sum(p["taken_at"] is None for p in photos)
print(f"사진 {len(photos)}장 · 동영상 {videos}개 · 읽기 실패 {len(failed)}장 · 촬영 시각 미상 {unknown}장")
for rel, why in failed:
    print("  실패:", rel, why)

rows = []
for run in range(1, 4):
    R._EMBEDDERS.clear()
    cache = f"/content/cache_run{run}"
    shutil.rmtree(cache, ignore_errors=True)
    a = R.analyze(photos, cache_dir=cache, drop_failed=True)
    t = a.timing
    rows.append((run, t["model_load_s"], t["decode_s"], t["decoded"], t["embed_s"], t["embedded"]))

print("\n| 회차 | 모델 로딩(s) | 디코딩 합계(s) | 추론 합계(s) | 1장당 디코딩(ms) | 1장당 추론(ms) |")
print("|---|---|---|---|---|---|")
for run, ml, ds, dn, es, en in rows:
    print(f"| {run} | {ml:.2f} | {ds:.2f} ({dn}장) | {es:.2f} ({en}장) | {ds / max(dn, 1) * 1000:.0f} | {es / max(en, 1) * 1000:.0f} |")

later = rows[1:]
load = statistics.fmean(r[1] for r in later)
per = statistics.fmean(r[2] / max(r[3], 1) + r[4] / max(r[5], 1) for r in later)
print(f"\n300장 환산 (2·3회차 평균): 모델 로딩 {load:.1f}s + 300 x {per * 1000:.0f}ms = 약 {load + 300 * per:.0f}초")

print("\n형식별 순차 디코딩 (정방향 512px, 1장씩)")
by_ext = {}
for p in photos:
    by_ext.setdefault(Path(p["path"]).suffix.lower(), []).append(p["path"])
for ext, paths in sorted(by_ext.items()):
    t0, ok = time.perf_counter(), 0
    for path in paths:
        try:
            J.open_upright(path, R.RANK_SIDE)
            ok += 1
        except Exception:
            pass
    print(f"  {ext}: {ok}장, 1장당 {(time.perf_counter() - t0) / max(ok, 1) * 1000:.0f}ms")

## 2. 기준값 점검

두 페이지를 내려받아 내 PC의 브라우저로 엽니다.

1. **`clusters.html`:** 시간 구간과 묶음, 기준값 근처의 경계 사례를 봅니다. 세로 사진이 바로 서 있는지도 여기서 확인합니다.
2. **`pairs.html`:** 연속 사진 쌍마다 [연사] 또는 [다른 장면]을 누릅니다. 떨어져 찍은 같은 장소 사진은 '직접 고르기'로 추가합니다.
   - **연사:** 한 장만 남겨도 되는 같은 컷
   - **다른 장면:** 같은 장소라도 둘 다 남겨야 하는 사진

   페이지 아래에 현재 기준값의 정답률, 기준값별 표, 틀린 쌍이 계산됩니다. 결과는 CSV로 저장합니다.

In [ ]:
!python curation.py "{DEV_ALBUM}" --cache /content/cache_run1 --work {OUT}
!python pair_check.py "{DEV_ALBUM}" --cache /content/cache_run1 --work {OUT}

from google.colab import files
dev_out = album_out_dir(OUT, DEV_ALBUM)
for name in ("clusters.html", "pairs.html"):
    files.download(str(dev_out / name))

**(선택) 다른 기준값으로 실제 묶음 다시 보기.** 기준값별 정답률은 `pairs.html`에 이미 있습니다. 이 셀은 실제 묶음 모양까지 보고 싶을 때만 씁니다. 기준값을 바꾸면 바꾸기 전후 값을 같이 기록합니다.

In [ ]:
TRY_SIM = 0.90
p = json.load(open("params.json"))
p["clip_sim"] = TRY_SIM
try_params = f"/content/params_{TRY_SIM}.json"
json.dump(p, open(try_params, "w"))
try_out = f"/content/out_sim{TRY_SIM}"
!python curation.py "{DEV_ALBUM}" --params {try_params} --cache /content/cache_run1 --work {try_out}
files.download(str(album_out_dir(try_out, DEV_ALBUM) / "clusters.html"))

## 3. 평가 앨범 정답 페이지

앨범마다 `label_*.html`을 만들어 앨범 주인에게 보냅니다.
1. 주인이 ★ 10장과 중요 장면을 표시하고 `owner_*.csv`를 저장합니다.
2. 받은 CSV를 Drive의 `owner` 폴더에 넣습니다. 이름은 그대로 둬도 됩니다.

**정답을 받기 전에는 추천 결과(`compare.html`)를 보여주지 않습니다.**

In [ ]:
for album_id, (folder, _) in EVAL.items():
    !python labeler.py "{folder}" --album-id {album_id} --cache /content/cache_eval --work {OUT}
    d = album_out_dir(OUT, folder)
    files.download(str(d / f"label_{d.name}.html"))

## 4. 추천 결과 예시

서버 연동을 확인할 때 쓰는 `recommend()` 결과입니다. photo_id는 앨범 기준 상대 경로입니다.

In [ ]:
!python recommend.py --folder "{TEST_ALBUM}" --cache /content/cache_test --out /content/result.json
files.download("/content/result.json")

## 5. 비교 평가

정답 파일(owner.csv)이 모두 모이고 `params.json`을 확정한 뒤 실행합니다. 출력의 `rule_version`을 결과와 함께 기록하고, 개발에 쓴 앨범은 넣지 않습니다.

정답 파일은 `owner` 폴더에 넣습니다. 이름을 `EVAL`의 경로와 맞추지 않아도, 정답 페이지에서 저장된 이름(`owner_<폴더 이름>_xxxxxx.csv`) 그대로 두면 자동으로 찾습니다. 파일이 하나라도 없으면 아무것도 계산하지 않고 안내만 출력합니다.

In [ ]:
def find_owner(folder, path):
    if os.path.exists(path):
        return path
    hits = glob.glob(f"{J1}/owner/owner_{Path(folder).name}_*.csv")
    return max(hits, key=os.path.getmtime) if hits else None

pairs, missing = [], []
for album_id, (folder, path) in EVAL.items():
    found = find_owner(folder, path)
    if found:
        print(f"{album_id}: {found}")
        pairs.append((folder, found))
    else:
        missing.append((album_id, path))

if missing:
    print("\n정답 파일이 아직 없어서 비교 평가를 건너뜁니다.")
    for album_id, path in missing:
        print(f"  없음: {album_id} -> {path}")
    print("3번 섹션의 정답 페이지에서 저장한 owner.csv를 owner 폴더에 넣은 뒤 다시 실행하세요.")
else:
    args = " ".join(f'"{a}" "{c}"' for a, c in pairs)
    !python evaluate.py {args} --params params.json --cache /content/cache_eval --out /content/eval.csv
    if os.path.exists("/content/eval.csv"):
        files.download("/content/eval.csv")

## 6. 초안 평가 (수용률 · 흔들림)

5번까지 끝낸 뒤 실행합니다. 이 페이지는 추천 결과를 보여주므로, **정답(owner.csv)을 받은 앨범에만** 씁니다.

1. 앨범마다 `review_*.html`을 만들어 앨범 주인에게 보냅니다. 제안 방식, 화질 점수를 끈 방식, 균등 추출의 초안을 섞어서, 어느 방식이 고른 사진인지 모르게 보여줍니다.
2. 주인이 사진마다 '건질 사진', '흔들림·흐림', '별로' 중 하나를 누르고 [저장]으로 `review_*.csv`를 받습니다.
3. 받은 CSV를 Drive의 `review` 폴더에 넣고 아래 두 번째 셀을 실행합니다.

In [ ]:
for album_id, (folder, _) in EVAL.items():
    !python draft_review.py "{folder}" --album-id {album_id} --params params.json --cache /content/cache_eval --work {OUT}
    d = album_out_dir(OUT, folder)
    files.download(str(d / f"review_{d.name}.html"))

In [ ]:
reviews = sorted(glob.glob(f"{J1}/review/review_*.csv"))
if reviews:
    args = " ".join(f'"{r}"' for r in reviews)
    !python draft_review.py --score {args}
else:
    print(f"{J1}/review 폴더에 review_*.csv가 아직 없습니다.")

## 7. 외부 검증: Photo Triage

연사 중 한 장 고르기를 남이 모은 사람 선호 데이터로 검증합니다. [Princeton/Adobe Photo Triage](https://pixl.cs.princeton.edu/pubs/Chang_2016_ATF/index.php)(Chang et al., SIGGRAPH 2016)는 개인 앨범의 비슷한 사진 묶음 5,953개(15,545장)에서, 사람들이 두 사진 중 어느 쪽을 더 좋아했는지 모은 데이터입니다.

- **받는 곳:** 원래 사이트가 닫혀서 Kaggle 복사본(`ericwolter/triage`, 약 13.6GB)을 Colab 서버로 바로 받습니다. 내 PC에는 받지 않습니다.
- **Kaggle 인증:** Kaggle의 Settings > API에서 토큰을 만들고, 첫 셀을 실행하면 나오는 입력 칸에 붙여 넣습니다(입력한 값은 화면에 보이지 않습니다). 예전 방식의 `kaggle.json`이 있으면 입력 칸을 비우고 Enter를 눌러 파일을 올려도 됩니다. 토큰은 비밀번호와 같으니 노트북 셀에 직접 적지 마세요.
- **라이선스:** 이미지는 연구용·비상업 라이선스입니다. 연구 대회 출품이라 연구 목적으로 쓰고, 이미지는 저장소나 Drive에 다시 올리지 않습니다.
- **시간:** 내려받기와 압축 풀기 10~20분, 특징 계산(약 1.3만 장) 20분 안팎입니다.
- **결과:** 선명도(지금 대표 컷 규칙), 미적 점수, 내용 점수, 그리고 이 데이터로 학습한 모델이 사람 선택을 맞힌 비율입니다. 학습에 쓰지 않은 묶음(train의 15%)과 공식 val에서 잽니다.
- **추천에 쓰기:** `triage_result.csv`와 `triage_ranker.npz`가 Drive의 `results`에 저장됩니다. 학습 모델이 두 시험 모두에서 선명도보다 확실히 나으면, `triage_ranker.npz`를 저장소의 `model/`에 넣고 `params.json`의 `rank_model`을 그 모델 이름(`small`, `prompts`, `clip`)으로 바꿉니다.

In [ ]:
import os
from getpass import getpass
if not os.environ.get("KAGGLE_API_TOKEN") and not os.path.exists("/root/.kaggle/kaggle.json"):
    token = getpass("Kaggle API 토큰 (kaggle.json을 올리려면 비워 두고 Enter): ").strip()
    if token:
        os.environ["KAGGLE_API_TOKEN"] = token  # 이 런타임에서만 쓰고 저장하지 않습니다
    else:
        print("kaggle.json을 올려 주세요.")
        up = files.upload()
        os.makedirs("/root/.kaggle", exist_ok=True)
        with open("/root/.kaggle/kaggle.json", "wb") as f:
            f.write(next(iter(up.values())))
        os.chmod("/root/.kaggle/kaggle.json", 0o600)
!pip install -q -U "kaggle>=1.8.0"
if not glob.glob("/content/triage/**/train_pairlist.txt", recursive=True):
    !kaggle datasets download -d ericwolter/triage -p /content/triage --unzip -q
print(glob.glob("/content/triage/**/*pairlist*.txt", recursive=True))

In [ ]:
!python triage_eval.py /content/triage --cache /content/cache_triage --out /content/triage_result.csv --save-ranker /content/triage_ranker.npz
!mkdir -p "{J1}/results" && cp /content/triage_result.csv /content/triage_ranker.npz "{J1}/results/"
files.download("/content/triage_result.csv")

## 결과 보관

In [ ]:
!mkdir -p "{J1}/results" && cp -r {OUT} "{J1}/results/"
print("저장:", f"{J1}/results/")